### Задание 1 (8 баллов + 0.5)

Скачайте вот этот датасет - ("MERA-evaluation/MERA", "multiq"). Так как обучать ничего не придется возьмите train split. Возьмите оттуда 25 примеров (случайных, а не первых).

Преобразуйте колонку instruction, заменив переменные вроде `{support_text}` на реальные значения из других колонок, чтобы получился готовый промпт (инпут для модели).

#### Метрика (2 балла)

Напишите две метрики для оценки правильных ответов: 1) exact match и 2) F1 по леммам. Для первой просто добавьте базовую предобработку вроде удаления пунктуации и регистра и просто сопоставьте строки между собой. Для второй к базовой предобработке добавьте токенизацию и лемматизацию, а поверх лемм рассчитайте F1 метрику (подсказка - precision можно рассчитать посчитав какой процент лемм в предсказанном ответе совпадает с правильным ответом).

Рассчитайте метрики на простом бейзлайне (например, можете попробовать взять самый частотный биграмм из промпта в качестве ответа). Можно получить +0.5 балла за неочевидный ненулевой бейзлайн.

#### Эксперимент 1 (2 балла)
Возьмите какую-нибудь не очень большую модель с huggingface (например, `Qwen/Qwen3.5-0.8B`, но можно и любую другую, которая умеет отвечать на инструкции и поддерживает thinking). Преобразуйте промпты из датасета в формат `messages = [{"role": "user", "content": prompt}]` с помощью чат темплейта этой модели.
Сгенерируйте ответы модели на полученных данных (25 примеров). На этом шаге не используйте thinking. Поставьте max_new_tokens на не очень большое значения, чтобы генерация не занимала слишком много времени.
Оцените качество ответов с помощью двух метрик.

#### Эксперимент 2 (1 балла)
К первому эксперименту добавьте какой-то system prompt, который по вашему должен помочь модели справится с результатом. Можете проанализировать ошибки из первого задания и написать инструкции на основе этого.
Сгенерируйте ответы с точно такими же параметрами как и в эксперименте 1.
Оцените качество ответов с помощью двух метрик.

#### Эсперимент 3 (3 балла)
На этом шаге включите thinking в модели и сгенерируйте предсказания (system prompt вы можете использовать по вашему желанию).
На этом шаге у вас будет несколько сложностей, которые нужно решить:
1) генерацию модели нельзя использовать напрямую, сначала нужно отделить thinking блок от самого ответа. Добавьте пост обработку
2) thinking может занять достаточно много токенов и его нужно ограничить, но при этом вам нужно, чтобы модель выдала и сам ответ, а не только незаконченные размышлений. Выберите max_new_tokens с рассчетом на (thinking_tokens + answer_tokens). Сначала запустите первую генерацию с thinking_tokens, затем проверьте, что в ответе содержится </think> тэг, который означает, что модель закончила думать (еще удобно использовать `stop_strings` в huggingface generation параметрах, чтобы заканчивать как только появился нужный токен). И если тэг есть, то просто запустите вторую генерацию с max_new_tokens (как продолжение уже накопившегося) или же поставьте </think> тэг вручную (обрубив размышление) и запустите вторую генерацию c answer_tokens.


In [3]:
from datasets import load_dataset
import pandas as pd
import string
from nltk.util import ngrams
from collections import Counter
import pymorphy3
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
mera = load_dataset("MERA-evaluation/MERA", "multiq", split='train').to_pandas()
random_mera = mera.sample(n=25, random_state=42)
random_mera

README.md:   0%|          | 0.00/168k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/2.87M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/3.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1056 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

,instruction,inputs,outputs,meta
260,Даны два текста:\nТекст 1: {support_text}\nТек...,{'text': 'Бобруйск — город областного подчинен...,Михаил Викторович Желудов,"{'id': 260, 'bridge_answers': 'Бобруйск'}"
832,Дан вопрос:\n{question}\nОтветьте на этот вопр...,{'text': 'Яуза — малая река в Московской облас...,Москвы,"{'id': 832, 'bridge_answers': 'Яузу'}"
846,Ниже приведены два текста.\nТекст 1: {support_...,"{'text': 'Ендырь — река в России, протекает в ...",Могилевской протоки Оби,"{'id': 846, 'bridge_answers': 'Ендырь'}"
1007,"Ответьте на вопрос ""{question}"". Для ответа и...",{'text': 'Эрик Дуглас Морли (26 сентября 1918 ...,Холборн,"{'id': 1007, 'bridge_answers': 'Эрику Морли'}"
88,Прочитайте два текста.\n1. {support_text}\n2. ...,{'text': 'Джордж Вашингтон Ламберт ( 13 сентя...,Санкт-Петербург,"{'id': 88, 'bridge_answers': 'Джорджа Вашингто..."
457,"Ответьте на вопрос ""{question}"". Для ответа и...",{'text': 'Лямин — река в Ханты-Мансийском авто...,Оби;Обь,"{'id': 457, 'bridge_answers': 'Лямин'}"
184,Текст 1: {support_text}\nТекст 2: {text}\nОпир...,{'text': 'Магнитогорск — город в Челябинской о...,Сергей Бердников,"{'id': 184, 'bridge_answers': 'Магнитогорск'}"
988,Прочитайте два текста.\n1. {support_text}\n2. ...,{'text': ''Кристал Пэлас' (полное название — Ф...,Премьер-лиге,"{'id': 988, 'bridge_answers': 'Кристал Пэлас'}"
367,"Ответьте на вопрос ""{question}"". Для ответа и...","{'text': 'Яченка — река в России, протекает в ...",Оки,"{'id': 367, 'bridge_answers': 'Яченки'}"
558,Прочитайте два текста.\n1. {support_text}\n2. ...,{'text': 'Сосьва (в верховье Большая Сосьва) —...,Тавда,"{'id': 558, 'bridge_answers': 'Сосьву'}"


In [5]:
print(random_mera["instruction"].iloc[0])

Даны два текста:
Текст 1: {support_text}
Текст 2: {text}
Опираясь на данные тексты, ответьте на вопрос: {question}
Ваш ответ не должен содержать дополнительные объяснения.
Ответ:


In [6]:
random_mera["ready_prompt"] = random_mera.apply(
    lambda row: row["instruction"].format_map(row["inputs"]), axis=1
)
random_mera["ready_prompt"] = random_mera["ready_prompt"].str.replace("\\n", "\n")

print(random_mera["ready_prompt"].iloc[0])

Даны два текста:
Текст 1: Сергей Анатольевич Широкий  ( род. 24 марта 1966, Бобруйск, Могилёвская область) — советский и белорусский футболист, игравший на позиции полузащитник.
Текст 2: Бобруйск — город областного подчинения в Белоруссии, административный центр Бобруйского района Могилёвской области. Седьмой по численности населения и площади город Белоруссии, наиболее крупный центр областного подчинения.

Органы власти.
Городской Совет депутатов 28 созыва был избран 18 февраля 2018 года. Председателем горсовета является Михаил Викторович Желудов. "Исполнительным и распорядительным органом власти" является "Бобруйский городской исполнительный комитет". Председатель назначается Президентом и утверждается депутатами горсовета. Председателем Бобруйского горисполкома является Александр Студнев.
Опираясь на данные тексты, ответьте на вопрос: Кто является главой администрации в городе, где родился Сергей Анатольевич Широкий?
Ваш ответ не должен содержать дополнительные объяснения.
Ответ:


In [7]:
morph = pymorphy3.MorphAnalyzer()

def normalization(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    return " ".join(text.split())

def lemmas(text):
    clean_text = normalization(text)
    words = clean_text.split()
    return [morph.parse(word)[0].normal_form for word in words]

def exact_match(prediction, gold_answers):
    return max(normalization(prediction) == normalization(g) for g in gold_answers)

def lemma_f1(prediction, gold_answers):
    def f1(pred, gold):
        p, g = lemmas(pred), lemmas(gold)
        common = sum((Counter(p) & Counter(g)).values())
        if common == 0:
            return 0.0
        precision = common / len(p)
        recall = common / len(g)
        return 2 * precision * recall / (precision + recall)
    return max(f1(prediction, g) for g in gold_answers)

def baseline(prompt):
    words = normalization(prompt).split()
    important_words = []
    for w in words:
        pos = morph.parse(w)[0].tag.POS
        if pos in ["NOUN", "ADJF", "ADJS"]:
            important_words.append(w)

    bi_grams = list(ngrams(important_words, 2))
    if not bi_grams:
        bi_grams = list(ngrams(words, 2))

    if not bi_grams:
        return ""

    return " ".join(Counter(bi_grams).most_common(1)[0][0])

em_results = []
f1_results = []

for idx, row in random_mera.iterrows():
    prompt = row["ready_prompt"]
    gold_text = row["outputs"]
    gold = [gold_text] if isinstance(gold_text, str) else gold_text
    pred = baseline(prompt)
    em_results.append(exact_match(pred, gold))
    f1_results.append(lemma_f1(pred, gold))

print(f"Exact Match: {sum(em_results) / len(em_results):.4f}")
print(f"Lemma F1: {sum(f1_results) / len(f1_results):.4f}")

Exact Match: 0.0000
Lemma F1: 0.0400


In [8]:
model_name = 'Qwen/Qwen3.5-0.8B'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype='auto',
    device_map='auto'
)

preds = []
for idx, row in random_mera.iterrows():
    prompt = row["ready_prompt"]
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=20,
            do_sample=False,
        )

    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    preds.append(response.strip())

em_model_results = []
f1_model_results = []

for idx, (_, row) in enumerate(random_mera.iterrows()):
    pred = preds[idx]

    gold_text = row["outputs"]
    gold = [gold_text] if isinstance(gold_text, str) else gold_text

    em_model_results.append(exact_match(pred, gold))
    f1_model_results.append(lemma_f1(pred, gold))

print(f"Exact Match (Qwen): {sum(em_model_results) / len(em_model_results):.4f}")
print(f"Lemma F1 (Qwen): {sum(f1_model_results) / len(f1_model_results):.4f}")

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Exact Match (Qwen): 0.0400
Lemma F1 (Qwen): 0.1060


In [9]:
preds_with_system = []

system_prompt = (
    "Роль: Точный поисковый робот.\n"
    "Задача: Найди ответ в тексте и выведи ТОЛЬКО его.\n"
    "Формат ответа: СТРОГО 1-3 слова. ТОЛЬКО конкретное имя, название, дата или число.\n"
    "Правило: Пиши ИСКЛЮЧИТЕЛЬНО финальный ответ. Любые приветствия, вводные слова, пояснения или комментарии ПОЛНОСТЬЮ исключены."
)

for idx, row in random_mera.iterrows():
    prompt = row["ready_prompt"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    preds_with_system.append(response.strip())

em_system_results = []
f1_system_results = []

for idx, (_, row) in enumerate(random_mera.iterrows()):
    pred = preds_with_system[idx]

    gold_text = row["outputs"]
    gold = [gold_text] if isinstance(gold_text, str) else gold_text

    em_system_results.append(exact_match(pred, gold))
    f1_system_results.append(lemma_f1(pred, gold))

print(f"\nРезультаты Qwen3.5-0.8B с System Prompt:")
print(f"Exact Match (System): {sum(em_system_results) / len(em_system_results):.4f}")
print(f"Lemma F1 (System): {sum(f1_system_results) / len(f1_system_results):.4f}")


Результаты Qwen3.5-0.8B с System Prompt:
Exact Match (System): 0.0800
Lemma F1 (System): 0.1467


In [13]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

system_prompt = (
    "Роль: Точный поисковый робот.\n"
    "Задача: Найди ответ в тексте и выведи ТОЛЬКО его.\n"
    "Формат ответа: СТРОГО 1-3 слова. ТОЛЬКО конкретное имя, название, дата или число.\n"
    "Правило: Сначала подробно распиши логику и рассуждения, а в самом конце выдай краткий ответ."
)

preds_with_thinking = []

for idx, row in random_mera.iterrows():
    prompt = row["ready_prompt"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        first_gen_ids = model.generate(
            **model_inputs,
            max_new_tokens=20,
            do_sample=False,
            stop_strings=["</think>"],
            tokenizer=tokenizer,
        )

    decoded_first_gen = tokenizer.decode(
        first_gen_ids[0], skip_special_tokens=False
    )

    if "</think>" not in decoded_first_gen:
        first_gen_tokens = first_gen_ids[0]
        think_close_tensor = torch.tensor(
            tokenizer.encode("\n</think>\n", add_special_tokens=False),
            device=model.device,
        )
        continued_inputs = torch.cat(
            [first_gen_tokens, think_close_tensor]
        ).unsqueeze(0)
    else:
        continued_inputs = first_gen_ids

    with torch.no_grad():
        final_gen_ids = model.generate(
            input_ids=continued_inputs,
            max_new_tokens=20,
            do_sample=False,
        )

    new_tokens = final_gen_ids[0][continued_inputs.shape[1] :]
    final_answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    preds_with_thinking.append(final_answer)

em_think_results = []
f1_think_results = []

for idx, (_, row) in enumerate(random_mera.iterrows()):
    pred = preds_with_thinking[idx]
    gold_text = row["outputs"]
    gold = [gold_text] if isinstance(gold_text, str) else gold_text

    em_think_results.append(exact_match(pred, gold))
    f1_think_results.append(lemma_f1(pred, gold))

print(f"\nРезультаты {model_name} с режимом Thinking:")
print(f"Exact Match (Thinking): {sum(em_think_results) / len(em_think_results):.4f}")
print(f"Lemma F1 (Thinking): {sum(f1_think_results) / len(f1_think_results):.4f}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


Результаты Qwen/Qwen3-0.6B с режимом Thinking:
Exact Match (Thinking): 0.0800
Lemma F1 (Thinking): 0.1512
